# ARIMA++

The purpose of this notebook is to run ARIMA, etc.

Below, we run:
- auto_arima. Ignore the fact that there are missing days.
    - this has worse errors than the naive baseline model
- sarimax with no seasonality and has fourier terms to account for seasonality
    - performed better than the baseline model
- sarimax with no seasonality, yes fourier terms, yes sunlight_duration exoenous variable
    - ran for 5 systems and performed worse than sarimax without seasonality

In [139]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn
from pathlib import Path
from sklearn.model_selection import TimeSeriesSplit
from extract_and_clean import Clean

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error as mse

#now more time-series specific things
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import statsmodels.tsa.api as sm
import pmdarima as pm
from pmdarima import auto_arima
import itertools
from statsmodels.tsa.statespace.sarimax import SARIMAX

system_ids = [2105, 2107, 7333, 9068] #the relevant ids
#system_id = 2105 #may have to look through them eventually
#path = f'../../../../data_ds_project/systems/prize/{system_id}/'

def systemPath(system_id):
    return f'../../../../data_ds_project/systems/prize/{system_id}/'

#Instead of using all the data/files for EDA, we will use only some
#we begin with a list of inverters, and then add on the meters, to make sure we're choosing from all of the possibilities
individual_data = pd.read_csv('../../../../data_ds_project/systems/prize/new_inverter_names_for_prize_cleaned_power.csv')
individual_data

meters_df = pd.DataFrame({
    'system_id' : [2105, 9068], 
    'old_name' : ['meter','meter'], 
    'new_name' : ['000','000']
})
all_datas = pd.concat([individual_data, meters_df], ignore_index = True)
for i in range(len(all_datas)):
    all_datas.iloc[i,2] = str(all_datas['new_name'][i]).zfill(3)
all_datas['file_name'] = (all_datas['system_id'].astype(str) + "_"
                          + np.where(all_datas['old_name'] == 'meter', 'meter', 'inverter') + "_"
                          + all_datas['new_name'].astype(str))
all_datas

sample_data = all_datas #update as needed

We need to figure out the optimal ARIMA constants. We know that seasonal = 365. We need to figre out (p,q,d) and (P,Q,D).

In [121]:
#First, run straight auto-ARIMA. No filling in the missing data points. This will be a baseline for using ARIMA.
# Why? Missing data shouldn't mess with *too* much -- only with appx as many data points as there are chunks missing

#will write this to run through ALL of the systems, though to test the code, it'll only run one.
sample_data = all_datas[:5]
all_errors = np.zeros(len(sample_data))
all_orders = []
for i in range(len(sample_data)):
    #file name
    file_name = all_datas.loc[i, 'file_name'] + ".csv"
    print(f"Now working on {file_name}")

    df = pd.read_csv(f'../../../../data_ds_project/systems/prize/prize_cleaned_power/{file_name}')
    #print(f"System_id = {str(sample_data.iloc[i,0])}")

    #turn strings into daytime
    df['time'] = pd.to_datetime(df['time'])

    max = df['power'].max()
    df['proportion'] = df['power']/max

    #train test split
    df_train = df.loc[:int(0.8*len(df))]
    df_test = df.loc[int(0.8*len(df)):]
    x_train = df_train['proportion']

    auto_model = auto_arima(df['proportion'], display = False)
    order = auto_model.order

    #now that we have the model, should do cross-validation
    n_val = 3
    errors_sys = np.zeros(n_val)
    kfold = TimeSeriesSplit(n_val)
    for j,(ind_train, ind_ho) in enumerate(kfold.split(x_train)):
        train = x_train.iloc[ind_train]
        ho = x_train.iloc[ind_ho]

        model = SARIMAX(
            train,
            order=order,
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        mod = model.fit()
        y_pred = mod.forecast(steps = len(ho))
        
        errors_sys[j] = mse(y_pred, ho)
    all_errors[i] = errors_sys.mean()
    all_orders.append(order)


print(all_errors)
print(all_orders)


Now working on 2107_inverter_000.csv
Now working on 2107_inverter_001.csv
Now working on 2107_inverter_002.csv
Now working on 2107_inverter_003.csv
Now working on 2107_inverter_004.csv
[0.03084541 0.04143531 0.04017069 0.06257677 0.03531425]
[(1, 1, 1), (0, 1, 2), (1, 1, 1), (1, 0, 2), (1, 1, 1)]


In [47]:
#save these errors *just in case* -- won't have to run the model again
naive_arima_errors = pd.DataFrame(all_errors)
naive_arima_errors.to_csv('errors/naive_arima_errors.csv')

Now we become less naive. In particular, quick plots of daily average data indicate that there is *some* seasonality. The season appears to be 1 year -- 365 days. However, this is an incredibly long season, so we cannot use the seasonality function of SARIMA/ARIMA. Moreover, auto_arima is no longer an option since we will have missing data.

We will therefore use SARIMAX with no seasonality and Fourier exogenous variables to model the seasonality.

### SARIMAX with Fourier exogenous variables only

In [75]:
#seasonality fourier variables
#We will have to find an optimal number of fourier variables. For now, we'll have 6 columns.
# We will also make these absurdly long; just trim them to the correct size when needed.

t = np.arange(5000)

fourier = np.column_stack([
    np.sin(2*np.pi*t/365.25),
    np.cos(2*np.pi*t/365.25),
    np.sin(2*np.pi*t*2/365.25),
    np.cos(2*np.pi*t*2/365.25),
    np.sin(2*np.pi*t*3/365.25),
    np.cos(2*np.pi*t*3/365.25)
])

eX = pd.DataFrame(fourier)


In [ ]:
#note: LOTS of warnings about non-convergence. A couple about non-invertible starting conditions.
all_errors_arima_with_exog = pd.DataFrame(columns = ["file_name", "model (p,d,q,k)", "error"])
sample_data = all_datas.iloc[110:]
for i in range(110, len(all_datas)):
    file_name = sample_data.loc[i, 'file_name'] + ".csv"
    print(f"Now working on {file_name}")

    df = pd.read_csv(f'../../../../data_ds_project/systems/prize/prize_cleaned_power/{file_name}')
        #print(f"System_id = {str(sample_data.iloc[i,0])}")

        #turn strings into daytime
    df['time'] = pd.to_datetime(df['time'])

    max = df['power'].max()
    df['proportion'] = df['power']/max

        #fill in empty times 
    obj = Clean()
    new_df = obj.fill_missing_days_na(df)
        #display(new_df)

        #should do a train/test split
    new_df_train = new_df.loc[:0.8*int(len(new_df))]
    y=pd.DataFrame(new_df_train['proportion'])

    #do cross validation
    n_split=3
    kfold = TimeSeriesSplit(n_split)

    #First, will do ONE parameter. Then can move on to a search.

    best_model = (-1,-1,-1,-1)
    best_avg_error = 2 
    for p in range(3): #3
        for d in range(2): #2
            for q in range(3): #3
                for k in range(3): #3
                    errors = np.zeros(n_split)
                    for j,(ind_train, ind_ho) in enumerate(kfold.split(y)):
                        y_train = y.iloc[ind_train]
                        eX_train = eX.iloc[ind_train,:2*(k+1)]
                        y_ho = y.iloc[ind_ho]
                        eX_ho = eX.iloc[ind_ho,:2*(k+1)]

                        model = SARIMAX(
                            y_train,
                            exog = eX_train,
                            order = (p,d,q)
                        ).fit()

                        y_ho['y_pred'] = model.forecast(len(y_ho), exog = eX_ho)
                        y_ho = y_ho.dropna()

                        errors[j] = mse(y_ho['y_pred'],y_ho['proportion'])

                    #print(f"({p},{d},{q},{k}) errors = {errors}")
                    current_error = errors.mean()
                    if(current_error<best_avg_error):
                        best_avg_error = current_error
                        best_model = (p,d,q,k)
    new_row = pd.DataFrame(
        [(file_name, best_model, best_avg_error)],
        columns=["file_name","model (p,d,q,k)", "error"]
    )
    all_errors_arima_with_exog = pd.concat(
        [all_errors_arima_with_exog, new_row],
        ignore_index=True
    )
all_errors_arima_with_exog.to_csv('errors/all_errors_arima_with_exog.110,end.csv')

In [ ]:
# #combine all files into one file. NO LONGER NEEDED.
# df1 = pd.read_csv("errors/all_errors_arima_with_exog.0.19.csv")
# df2 = pd.read_csv("errors/all_errors_arima_with_exog.20.109.csv")
# df3 = pd.read_csv("errors/all_errors_arima_with_exog.110,end.csv")
# df4 = pd.concat([df1, df2, df3], ignore_index = True)
# df4 = df4.drop(columns = ["Unnamed: 0"])
# df4
# df4.to_csv("errors/arima_with_fourier_only.csv", index = False)

In [ ]:
df_sarimax = pd.read_csv("errors/arima_with_fourier_only.csv")
df_sarimax['error'].mean()

np.float64(0.03111282546841732)

In [118]:
df_autoarima = pd.read_csv("errors/naive_arima_errors.csv")
df_autoarima['0'].mean()

np.float64(0.0747627844044867)

## ARIMA with fourier and exogenous variables

We now aim to add some exogenous variables --- in particular, weather-related. We will stick with number of sunlight hours. 

Note that this will likely change the seasonality portion as well, since the amount of sunlight will likely not be enough (also need the angle to the earth, which is determined by time of year)

In [133]:
#re-inputting as a reminder

#seasonality fourier variables
#We will have to find an optimal number of fourier variables. For now, we'll have 6 columns.
# We will also make these absurdly long; just trim them to the correct size when needed.

t = np.arange(5000)

fourier = np.column_stack([
    np.sin(2*np.pi*t/365.25),
    np.cos(2*np.pi*t/365.25),
    np.sin(2*np.pi*t*2/365.25),
    np.cos(2*np.pi*t*2/365.25),
    np.sin(2*np.pi*t*3/365.25),
    np.cos(2*np.pi*t*3/365.25)
])

fourier = pd.DataFrame(fourier)


In [137]:
#will now run SARIMA with exogenous variables again
#do for 5 systems -- see if errors are any better. Takes several times longer to run.

#note: LOTS of warnings about non-convergence. A couple about non-invertible starting conditions.
all_errors_arima_with_exog = pd.DataFrame(columns = ["file_name", "model (p,d,q,k)", "error"])
sample_data = all_datas.iloc[:5]
for i in range(len(sample_data)):
    file_name = sample_data.loc[i, 'file_name'] + ".csv"
    print(f"Now working on {file_name}")

    df = pd.read_csv(f'../../../../data_ds_project/systems/prize/prize_cleaned_power/{file_name}')
        #print(f"System_id = {str(sample_data.iloc[i,0])}")
    system_id = str(sample_data.iloc[i,0])

        #turn strings into daytime
    df['time'] = pd.to_datetime(df['time'])

    max = df['power'].max()
    df['proportion'] = df['power']/max

        #fill in empty times 
    obj = Clean()
    new_df = obj.fill_missing_days_na(df)
        #display(new_df)

        #should do a train/test split
    new_df_train = new_df.loc[:0.8*int(len(new_df))]
    y=pd.DataFrame(new_df_train['proportion'])

    #make a dataframe of exogenous variables. It will consist of Fourier followed by a column from prize_weather/{system_id}.csv
    eX = fourier
    weather_df = pd.read_csv(f"prize_weather/{system_id}.csv")
    eX['sunshine_duration'] = weather_df['sunshine_duration']
    #display(eX)

    #do cross validation
    n_split=3
    kfold = TimeSeriesSplit(n_split)

    #search for best parameters
    best_model = (-1,-1,-1,-1)
    best_avg_error = 2 
    for p in range(3): #3
        for d in range(2): #2
            for q in range(3): #3
                for k in range(3): #3
                    errors = np.zeros(n_split)
                    for j,(ind_train, ind_ho) in enumerate(kfold.split(y)):
                        y_train = y.iloc[ind_train]
                        cols_to_select = np.r_[0:2*(k+1), 6]
                        eX_train = eX.iloc[ind_train,cols_to_select]
                        y_ho = y.iloc[ind_ho]
                        eX_ho = eX.iloc[ind_ho,cols_to_select]

                        model = SARIMAX(
                            y_train,
                            exog = eX_train,
                            order = (p,d,q)
                        ).fit()

                        y_ho['y_pred'] = model.forecast(len(y_ho), exog = eX_ho)
                        y_ho = y_ho.dropna()

                        errors[j] = mse(y_ho['y_pred'],y_ho['proportion'])

                    #print(f"({p},{d},{q},{k}) errors = {errors}")
                    current_error = errors.mean()
                    if(current_error<best_avg_error):
                        best_avg_error = current_error
                        best_model = (p,d,q,k)
    new_row = pd.DataFrame(
        [(file_name, best_model, best_avg_error)],
        columns=["file_name","model (p,d,q,k)", "error"]
    )
    all_errors_arima_with_exog = pd.concat(
        [all_errors_arima_with_exog, new_row],
        ignore_index=True
    )
#all_errors_arima_with_exog.to_csv('errors/all_errors_arima_with_exog.110,end.csv')
all_errors_arima_with_exog

Now working on 2107_inverter_000.csv


c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed t

Now working on 2107_inverter_001.csv


c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed t

Now working on 2107_inverter_002.csv


...

Now working on 2107_inverter_003.csv


c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed t

Now working on 2107_inverter_004.csv


c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed t

,file_name,"model (p,d,q,k)",error
0,2107_inverter_000.csv,"(2, 0, 1, 2)",0.037671
1,2107_inverter_001.csv,"(0, 0, 0, 1)",0.04335
2,2107_inverter_002.csv,"(0, 0, 0, 2)",0.046836
3,2107_inverter_003.csv,"(0, 0, 0, 2)",0.037619
4,2107_inverter_004.csv,"(0, 0, 0, 2)",0.045742


In [ ]:
all_errors_arima_with_exog
#note that this is WORSE than without the sunlight_duration data! 
#We will not run the remainder of the systems since the runtime is very long.

,file_name,"model (p,d,q,k)",error
0,2107_inverter_000.csv,"(2, 0, 1, 2)",0.037671
1,2107_inverter_001.csv,"(0, 0, 0, 1)",0.04335
2,2107_inverter_002.csv,"(0, 0, 0, 2)",0.046836
3,2107_inverter_003.csv,"(0, 0, 0, 2)",0.037619
4,2107_inverter_004.csv,"(0, 0, 0, 2)",0.045742
